# ORC LLM DataFrame Runner

Run this notebook from the project root. Edit the configuration cell, then run the remaining cells top to bottom.

Inference is local. No external model/API calls are made. If the GGUF file is missing, the notebook downloads it from the GitHub Release asset configured in `scripts/download_model.py`.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
MODEL_PATH = PROJECT_ROOT / "models" / "LFM2.5-1.2B-Instruct-Q4_K_M.gguf"

# Set INPUT_CSV to your data file path, or leave as None to use the starter DataFrame below.
INPUT_CSV = None
TEXT_COLUMN = "notes"
OUTPUT_COLUMN = "policy_result"
OUTPUT_CSV = "orc_llm_output.csv"
SAVE_OUTPUT_CSV = True

PROMPT = """Classify this note as one of:
- meets_policy
- violates_policy
- needs_review

Note:
{text}

Return only the label."""

MAX_TOKENS = 32

# RTX 2050 / CPU-friendly defaults. Use n_gpu_layers=0 for CPU-only.
N_CTX = 4096
N_GPU_LAYERS = -1
N_BATCH = 512
N_UBATCH = 512

In [ ]:
import subprocess
import sys

sys.path.insert(0, str(PROJECT_ROOT / "src"))

if not MODEL_PATH.exists():
    subprocess.run([sys.executable, "scripts/download_model.py"], check=True)

if not MODEL_PATH.exists():
    raise FileNotFoundError(f"Model was not downloaded: {MODEL_PATH}")

MODEL_PATH

In [ ]:
import pandas as pd

from orc_llm import apply_prompt_to_column, benchmark_tokens_per_second, load_model

model = load_model(
    model_path=MODEL_PATH,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    n_batch=N_BATCH,
    n_ubatch=N_UBATCH,
)

print("Model loaded:", MODEL_PATH)

In [ ]:
if INPUT_CSV:
    df = pd.read_csv(INPUT_CSV)
else:
    df = pd.DataFrame(
        {
            "id": [1, 2],
            "notes": [
                "Customer asked for a refund after the warranty expired.",
                "Customer shared payment card details in plain text.",
            ],
        }
    )

if TEXT_COLUMN not in df.columns:
    raise ValueError(f"TEXT_COLUMN={TEXT_COLUMN!r} not found. Available columns: {list(df.columns)}")

df.head()

In [ ]:
result = apply_prompt_to_column(
    df,
    text_column=TEXT_COLUMN,
    output_column=OUTPUT_COLUMN,
    prompt=PROMPT,
    model=model,
    max_tokens=MAX_TOKENS,
    progress=True,
)

if SAVE_OUTPUT_CSV:
    result.to_csv(OUTPUT_CSV, index=False)
    print(f"Saved: {OUTPUT_CSV}")

result

In [ ]:
benchmark_tokens_per_second(model=model, max_tokens=32)